In [ ]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/mnt/c/users/sduai/documents/projis/research/nids_dl/nids-research")
sys.path.insert(0, str(ROOT / "src"))

import torch


from nids_dl import RFConfig, TrainConfig, evaluate, extract_features, fit_rf, train_extractor
from nids_dl.data import load_processed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
  print(f"Using {torch.cuda.device_count()} GPU(s):")
  for i in range(torch.cuda.device_count()):
      print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}")
else:
  print("No GPU found, using CPU")


In [ ]:
DEVICE

In [ ]:
ROOT

## 1. Load preprocessed NSL-KDD

In [ ]:
train = load_processed(ROOT / "data" / "processed" / "train.pt")
test = load_processed(ROOT / "data" / "processed" / "test.pt")

X_tr, y_tr = train["X"], train["y_bin"]
X_te, y_te = test["X"], test["y_bin"]
X_tr.shape, X_te.shape, int(y_tr.max().item()) + 1

## 2. Phase 1 — train the DL feature extractor

Cross-entropy on the temporary softmax head, Adam, with the MCL prediction-error-filter constraint re-applied after every step.

In [ ]:
cfg = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="binary", log_every=0, seed=0)
extractor, history = train_extractor(X_tr, y_tr, cfg, X_val=X_te, y_val=y_te)
history[-1]

In [ ]:
import pandas as pd

pd.DataFrame(history)

## 3. Extract features and fit RandomForest

In [ ]:
Fe_tr = extract_features(extractor, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te = extract_features(extractor, X_te, batch_size=512, device=DEVICE).numpy()
Fe_tr.shape, Fe_te.shape

In [ ]:
clf = fit_rf(Fe_tr, y_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_train = evaluate(clf, Fe_tr, y_tr.numpy())
metrics_test = evaluate(clf, Fe_te, y_te.numpy())
{
    "train": {k: metrics_train[k] for k in ("accuracy", "precision", "recall", "f1")},
    "test": {k: metrics_test[k] for k in ("accuracy", "precision", "recall", "f1")},
}

In [ ]:
print(metrics_test["report"])
metrics_test["confusion_matrix"]

## 4. Multi-class variant (5 classes: Normal/DoS/Probe/R2L/U2R)

In [ ]:
ym_tr, ym_te = train["y_mul"], test["y_mul"]
cfg_m = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="multi", seed=0)
extractor_m, history_m = train_extractor(X_tr, ym_tr, cfg_m, X_val=X_te, y_val=ym_te)
Fe_tr_m = extract_features(extractor_m, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te_m = extract_features(extractor_m, X_te, batch_size=512, device=DEVICE).numpy()
clf_m = fit_rf(Fe_tr_m, ym_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_te_m = evaluate(clf_m, Fe_te_m, ym_te.numpy())
print(metrics_te_m["report"])
{k: metrics_te_m[k] for k in ("accuracy", "precision", "recall", "f1")}